# Model Training and Experimentation

Interactive notebook for training and experimenting with models.

In [ ]:
import sys
sys.path.append('..')

import torch
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

from src.data.dataset import SmellDataset
from src.models.neobert_model import SmellToMoleculeModel
from src.training.trainer import Trainer

In [ ]:
# Configuration
config = {
    'model_name': 'bert-base-uncased',
    'num_chemicals': 300,
    'batch_size': 8,
    'learning_rate': 2e-5,
    'epochs': 5
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load data
tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
train_dataset = SmellDataset('../data/processed/train.csv', tokenizer)
val_dataset = SmellDataset('../data/processed/val.csv', tokenizer)

train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'])

In [ ]:
# Initialize model
model = SmellToMoleculeModel(
    model_name=config['model_name'],
    num_chemicals=config['num_chemicals']
)
model.to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train
trainer = Trainer(model, train_loader, val_loader, config)
history = trainer.train(epochs=config['epochs'])

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'], label='Val')
plt.legend()
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history.get('val_f1', []), label='F1')
plt.legend()
plt.title('Validation F1')
plt.tight_layout()
plt.show()